# M1: Baseline Pencarian Lintas Modal

**Kelompok 03 · IF25-40304 Pembelajaran Mesin Multimodal**

Notebook ini menjalankan evaluasi baseline pencarian lintas modal menggunakan encoder CLIP yang telah dipratih dan dibekukan (*frozen*), tanpa proses pelatihan. Model yang digunakan adalah `clip-ViT-B-32` dan `clip-ViT-B-32-multilingual-v1`.

**Dataset:** SEA-VL Indonesian subset, CC BY-SA 4.0 (Cahyawijaya et al., 2025).

## 1. Hubungkan Google Drive

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Instal dependensi

In [2]:
%pip install -q sentence-transformers

## 3. Konfigurasi lokasi dataset dan hasil

Pastikan file `bundle-mahasiswa.zip` berada di folder `Dataset-Tubes-Multimodal` pada Google Drive. Sesuaikan `DRIVE_DIR` apabila lokasi penyimpanannya berbeda.

In [3]:
DRIVE_DIR = "/content/drive/MyDrive/Dataset-Tubes-Multimodal"
ZIP_PATH = os.path.join(DRIVE_DIR, "bundle-mahasiswa.zip")

DATA_DIR = "/content/data/bundle-mahasiswa"
OUT_DIR = os.path.join(DRIVE_DIR, "results", "m1")

## 4. Siapkan dataset

In [4]:
os.makedirs("/content/data", exist_ok=True)

if not os.path.isdir(DATA_DIR):
    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(
            f"Dataset ZIP tidak ditemukan: {ZIP_PATH}. "
            "Periksa kembali lokasi file di Google Drive."
        )
    !unzip -q "{ZIP_PATH}" -d /content/data

required_files = ["train.csv", "val.csv"]
missing_files = [
    name for name in required_files
    if not os.path.isfile(os.path.join(DATA_DIR, name))
]
if missing_files:
    raise FileNotFoundError(
        f"File dataset tidak ditemukan di {DATA_DIR}: {missing_files}"
    )

os.makedirs(OUT_DIR, exist_ok=True)

print("Dataset berhasil disiapkan.")
print("Isi folder:", sorted(os.listdir(DATA_DIR)))
print("Lokasi hasil:", OUT_DIR)

Dataset berhasil disiapkan.
Isi folder: ['.omc', 'ATTRIBUTION.md', 'README.txt', 'images', 'train.csv', 'val.csv']
Lokasi hasil: /content/drive/MyDrive/Dataset-Tubes-Multimodal/results/m1


## 5. Parameter baseline

In [5]:
SEED = 42
KS = (1, 5, 10)
MAP_K = 10
ARAH = ["gambar->teks-ID", "gambar->teks-EN", "teks-ID->gambar", "teks-EN->gambar"]

## 6. Fungsi pendukung

In [6]:
import os, re, json, random
import numpy as np, pandas as pd
from PIL import Image


def norm(s):
    return re.sub(r"[^\w\s]", "", str(s).lower()).strip()


def ranks_of_truth(A, B):
    """Peringkat (1 = terbaik) pasangan benar A[i]<->B[i] di antara semua B."""
    sim = A @ B.T
    diag = np.diag(sim)[:, None]
    return (sim > diag).sum(1) + 1


def metrik(A, B):
    """Recall@K dan mAP@K; satu item relevan per query, jadi AP@K = 1/rank jika rank<=K."""
    r = ranks_of_truth(A, B)
    out = {f"Recall@{k}": float((r <= k).mean()) for k in KS}
    out[f"mAP@{MAP_K}"] = float(np.where(r <= MAP_K, 1.0 / r, 0.0).mean())
    return out, r

In [7]:
def muat_gambar(df, root):
    imgs = []
    for f in df["file"]:
        for p in (f, os.path.join(root, f), os.path.join(root, "images", os.path.basename(f))):
            if os.path.exists(p):
                imgs.append(Image.open(p).convert("RGB")); break
        else:
            raise FileNotFoundError(f)
    return imgs


def embed(df, root, cache):
    paths = {k: os.path.join(cache, f"{k}.npy") for k in ("img", "ti", "te")}
    if all(os.path.exists(p) for p in paths.values()):
        return {k: np.load(p) for k, p in paths.items()}
    from sentence_transformers import SentenceTransformer
    img_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32")
    txt_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1")
    E = img_model.encode(muat_gambar(df, root), batch_size=32, normalize_embeddings=True)
    Ti = txt_model.encode(list(df["caption_id"]), batch_size=64, normalize_embeddings=True)
    Te = img_model.encode(list(df["caption_en"]), batch_size=64, normalize_embeddings=True)
    os.makedirs(cache, exist_ok=True)
    for k, v in zip(("img", "ti", "te"), (E, Ti, Te)):
        np.save(paths[k], v)
    return {"img": E, "ti": Ti, "te": Te}

In [8]:
def evaluasi(emb, mask=None):
    E, Ti, Te = emb["img"], emb["ti"], emb["te"]
    if mask is not None:
        E, Ti, Te = E[mask], Ti[mask], Te[mask]
    pasangan = {ARAH[0]: (E, Ti), ARAH[1]: (E, Te), ARAH[2]: (Ti, E), ARAH[3]: (Te, E)}
    hasil, ranks = {}, {}
    for nama, (A, B) in pasangan.items():
        hasil[nama], ranks[nama] = metrik(A, B)
    return pd.DataFrame(hasil).T, ranks


def contoh_query(df, emb, ranks_teks_id_ke_gambar, n=10, min_gagal=2, topk=3):
    sim = emb["ti"] @ emb["img"].T
    top = np.argsort(-sim, axis=1)[:, :topk]
    gagal = [i for i in range(len(df)) if top[i, 0] != i]
    rng = random.Random(SEED)
    pilih = rng.sample(gagal, min(min_gagal, len(gagal)))
    sisa = [i for i in range(len(df)) if i not in pilih]
    pilih += rng.sample(sisa, n - len(pilih))
    rng.shuffle(pilih)
    baris = []
    for i in pilih:
        baris.append({
            "query_id": df["caption_id"].iloc[i],
            "benar": os.path.basename(df["file"].iloc[i]),
            "top1": os.path.basename(df["file"].iloc[top[i, 0]]),
            "top2": os.path.basename(df["file"].iloc[top[i, 1]]),
            "top3": os.path.basename(df["file"].iloc[top[i, 2]]),
            "top1_benar": bool(top[i, 0] == i),
            "peringkat_benar": int(ranks_teks_id_ke_gambar[i]),
            "kategori": df["category"].iloc[i] if "category" in df else "",
        })
    return pd.DataFrame(baris)

## 7. Muat data dan hitung embedding

In [9]:
np.random.seed(SEED); random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

val = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
N = len(val)
emb = embed(val, DATA_DIR, os.path.join(OUT_DIR, "cache_val"))
print(f"N val = {N}; pembanding acak Recall@1 = {1/N:.4f}")

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

0_CLIPModel/model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

0_CLIPModel/model.safetensors: downloading bytes:           |  0.00B            

... (19 outputs left out)

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.57MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

N val = 275; pembanding acak Recall@1 = 0.0036


## 8. Recall@K dan mAP@K (empat arah)

In [10]:
tabel, ranks = evaluasi(emb)
tabel.to_csv(os.path.join(OUT_DIR, "tabel_recall_val.csv"), float_format="%.4f")
tabel.round(4)

,Recall@1,Recall@5,Recall@10,mAP@10
gambar->teks-ID,0.2145,0.3745,0.5055,0.2930
gambar->teks-EN,0.4473,0.7127,0.8327,0.5594
teks-ID->gambar,0.1964,0.4182,0.5055,0.2830
teks-EN->gambar,0.3818,0.6727,0.7709,0.5073


## 9. Sensitivitas leakage

Tabel yang sama setelah membuang baris `val` yang caption-nya juga ditemukan di `train`.

In [11]:
tr = set(norm(c) for c in train["caption_id"]) | set(norm(c) for c in train["caption_en"])
bocor = np.array([(norm(i) in tr) or (norm(e) in tr)
                  for i, e in zip(val["caption_id"], val["caption_en"])])
print(f"Baris val dengan caption yang juga ada di train: {int(bocor.sum())}")
if bocor.sum():
    t2, _ = evaluasi(emb, mask=~bocor)
    t2.to_csv(os.path.join(OUT_DIR, "tabel_recall_val_tanpa_overlap.csv"), float_format="%.4f")
    display(t2.round(4))

Baris val dengan caption yang juga ada di train: 12


,Recall@1,Recall@5,Recall@10,mAP@10
gambar->teks-ID,0.2129,0.3802,0.5057,0.2922
gambar->teks-EN,0.4487,0.7186,0.8365,0.5618
teks-ID->gambar,0.1977,0.4259,0.5133,0.2868
teks-EN->gambar,0.3916,0.6692,0.7681,0.5130


## 10. Language gap (Recall@1, Inggris dikurangi Indonesia)

In [12]:
gap = {"gambar->teks": tabel.loc[ARAH[1], "Recall@1"] - tabel.loc[ARAH[0], "Recall@1"],
       "teks->gambar": tabel.loc[ARAH[3], "Recall@1"] - tabel.loc[ARAH[2], "Recall@1"]}
json.dump(gap, open(os.path.join(OUT_DIR, "language_gap.json"), "w"), indent=2)
{k: round(v, 4) for k, v in gap.items()}

{'gambar->teks': np.float64(0.2327), 'teks->gambar': np.float64(0.1855)}

## 11. Sepuluh contoh query Indonesia (minimal dua gagal)

In [13]:
ex = contoh_query(val, emb, ranks[ARAH[2]])
ex.to_csv(os.path.join(OUT_DIR, "contoh_query_id.csv"), index=False)
ex

,query_id,benar,top1,top2,top3,top1_benar,peringkat_benar,kategori
0,"Kota Tua Jakarta, yang secara resmi dikenal se...",svl_143496710.jpg,svl_144425430.jpg,svl_141004410.jpg,svl_143131921.jpg,False,18,wayang/seni
1,Bakso adalah hidangan sup khas Indonesia yang ...,svl_144425048.jpg,svl_144424955.jpg,svl_147562505.jpg,svl_142644506.jpg,False,13,kuliner
2,halaman di tengah Fort Marlborough,svl_140084276.jpg,svl_140084276.jpg,svl_147562946.jpg,svl_140084565.jpg,True,1,lain-lain
3,"Bedug, salah satu bukti akulturasi budaya Indo...",svl_140084842.jpg,svl_140084783.jpg,svl_143496795.jpg,svl_143497363.jpg,False,40,lain-lain
4,Ini adalah kemeja Batik. Batik berasal dari In...,svl_143496870.jpg,svl_140084373.jpg,svl_143496874.jpg,svl_140084768.jpg,False,20,pakaian/tekstil
5,Ini merupakan martabak keju dengan susu kental...,svl_142113138.jpg,svl_147562476.jpg,svl_147562458.jpg,svl_143496852.jpg,False,148,kuliner
6,Dewi Shinta pada tari Kecak di Pura Uluwatu,svl_141004111.jpg,svl_141004074.jpg,svl_140084416.jpg,svl_140084661.jpg,False,97,tarian
7,Pemeran Buto Cakil berfoto dengan 2 penari Man...,svl_140084705.jpg,svl_140084701.jpg,svl_140084768.jpg,svl_141004320.jpg,False,19,tarian
8,Aneka gorengan dengan sambal dan serundeng,svl_147525876.jpg,svl_144425061.jpg,svl_147563031.jpg,svl_140084024.jpg,False,49,kuliner
9,"Rempeyek atau peyek kacang, salah satu jenis k...",svl_147525769.jpg,svl_144923822.jpg,svl_147562476.jpg,svl_147562944.jpg,False,80,lain-lain
